## IMPACT DE LA MISE EN SERVICE D'UNE NOUVELLE STATION DE METRO SUR LE PRIX DE L'IMMOBILIER
### Projet 2A - Python pour la data science 
*Sonny Augusto, Abel Cornet-Carlos et Mayténa Labinsky*

## INTRODUCTION
Rennes est l'une des plus petites villes au monde à disposer d'un réseau de métro automatique. Parmi les nombreux critères qui déterminent le prix d'un bien immobilier, l'accessibilité aux transports en commun est un facteur différenciant. Dans ce contexte, ce projet analyse dans quelle mesure la mise en service d'une nouvelle ligne de métro impacte la valeur de l'immobilier des quartiers desservis. Notre objectif est de quantifier précisément la plus-value immobilière (en pourcentage) générée par la proximité immédiate d'une nouvelle station.

## Sommaire
- [Installation](#installation)
- [Préparation des Données](#préparation-des-données)
- [Statistiques Descriptives](#statistiques-descriptives)
- [Modèle de prédiction de la plus-value](#modèle-de-prédiction-du-prix)
- [Conclusion](#conclusion)

## Installation

In [ ]:
!pip install -r requirements.txt
#Modules:
import geopandas as gpd
import pandas as pd
import numpy as np
from matplotlib.colors import TwoSlopeNorm
#Fonctions:
import src.clear_data as cd
import src.get_data as gd
import src.analyse_data as ad
from src.model import preparer_et_entrainer, predire_impact_nouvelle_station, carte_plus_value
import matplotlib.pyplot as plt
import seaborn as sns
import mapclassify


# Préparation des données

### Adresses

In [2]:
url_metro = "https://data.rennesmetropole.fr/explore/dataset/topologie-des-points-darret-de-metro-du-reseau-star/download/?format=geojson"

url_dvf2021 = "https://files.data.gouv.fr/geo-dvf/latest/csv/2021/full.csv.gz"

url_dvf2022 = "https://files.data.gouv.fr/geo-dvf/latest/csv/2022/full.csv.gz"

url_dvf2023 = "https://files.data.gouv.fr/geo-dvf/latest/csv/2023/full.csv.gz"

url_dvf2024 = "https://files.data.gouv.fr/geo-dvf/latest/csv/2024/full.csv.gz"

url_dvf2025 = "https://files.data.gouv.fr/geo-dvf/latest/csv/2025/full.csv.gz"

### Import et modifications

In [ ]:
#Import
df_dvf_raw2021 = gd.fetch_dvf_api(url_dvf2021)
df_dvf_raw2022 = gd.fetch_dvf_api(url_dvf2022)
df_dvf_raw2023 = gd.fetch_dvf_api(url_dvf2023)
df_dvf_raw2024 = gd.fetch_dvf_api(url_dvf2024)
df_dvf_raw2025 = gd.fetch_dvf_api(url_dvf2025)


In [ ]:
gdf_metro_raw = gd.fetch_metro_api(url_metro)

Analyse des bases de données en vu de les nettoyer

Analyse des 5 dvfs en vue de la fusion:
On regarde si elles contiennent bien les mêmes colonnes



In [ ]:
list_dvf = [
    df_dvf_raw2021, 
    df_dvf_raw2022, 
    df_dvf_raw2023, 
    df_dvf_raw2024, 
    df_dvf_raw2025
]
labels = ["2021", "2022", "2023", "2024", "2025"]
ad.verify_dvf_columns(list_dvf, labels)


Une fois qu'on a vu que les colonnes étaient identiques, on les mets bout à bout.

In [6]:
#fusion
df_dvf_raw = cd.merge_yearly_dvf(list_dvf)
df_dvf_raw.to_csv("data/merged_raw.csv", index=False)

### Nettoyage de la base


Pour faciliter l'analyse des données dans la suite, on nettoie les deux bases de données: on enlève les lignes contenant des valeurs manquantes, on ne garde dans la base dvf que les colonnes utiles pour l'analyse, et que les ventes d'appartements et de maisons, on rajoute aussi le prix au m² pour faciliter les analyses.

In [7]:
#Nettoyage (projection, gestion des valeurs manquantes et sélection des colonnes utiles et rajout du prix/m²)
gdf_metro = cd.clean_metro_data(gdf_metro_raw)
gdf_dvf = cd.clean_dvf_data(df_dvf_raw)
gdf_dvf['prix_m2'] = gdf_dvf['valeur_fonciere'] / gdf_dvf['surface_reelle_bati']

### Analyse et suppression des valeurs extrêmes

Pour ne pas que les analyses soient faussées par des vente de 0 m² ou des prix trop élevés, on enlève de la base de donnée les valeurs extrêmes du prix au m² et de la superficie.

In [8]:
seuils_surface=ad.extreme_value_surface(gdf_dvf)

In [9]:
gdf_dvf=cd.remove_extreme_values(gdf_dvf, "surface_reelle_bati", seuils_surface[0], seuils_surface[1])

In [10]:
seuils_prix=ad.extreme_value_prix(gdf_dvf)

In [11]:
gdf_dvf=cd.remove_extreme_values(gdf_dvf, "prix_m2", seuils_prix[0], seuils_prix[1])

On fusionne maintenant les bases pour obtenir une seule base, avec à chaque ligne la distance à la station de métro la plus proche pour la ligne a et pour la ligne b.

In [12]:
#Base fusionnée
gdf_final = cd.merge_dvf_by_line(gdf_dvf, gdf_metro)

# Statistiques Descriptives 

Cette section a pour objectif d'explorer le marché immobilier rennais avant l'application de notre modèle.

In [ ]:
print(gdf_final.describe())

In [ ]:
import src.stats_desc as sd

# Statistiques générales
print("--- Statistiques Globales ---")
print(sd.get_general_stats(gdf_final))

# Comparaison Ligne A vs Ligne B
print("\n--- Comparaison par Ligne de Métro ---")
stats_lignes = sd.get_stats_by_ligne(gdf_final)
print(stats_lignes)

# Statistique par tranche (distance min au métro <250, 250-500, 500-800, >800)
print("\n--- Statistique par tranche ---")
stats_tranches = sd.analyse_prix_dist_tranche(gdf_final)
print(stats_tranches)

#Graphique baton des statistiques par tranche
sd.plot_prix_par_tranche(stats_tranches)

# Statistique par variable de contrôle
print("\n--- Statistique par variable de contrôle (maison/appart) ---")
var_control_type = sd.compare_proximity_controlled(gdf_final)
print(var_control_type)


On constate que le prix de l'immobilier à moins de 250 mètres d'une station n'est pas supérieur à celui des autres trannches de distance, ce qui contredit notre intuition initiale.

In [ ]:
plt.figure(figsize=(10, 8))
sns.scatterplot(data=gdf_final, x="longitude", y="latitude", hue="prix_m2", 
                palette="YlOrRd", size="prix_m2", sizes=(10, 100), alpha=0.6)
plt.title("Répartition géographique des prix au m² à Rennes")
plt.show()

### Différence-en-différences

Nous appliquons la méthode de la différence-en-différences (DiD), pour visualiser l'évolution des prix  pour deux groupes :
* le groupe traité : immobilier situé à moins de 250m d'une station.
* le groupe de contrôle: immobilier situé au-delà de 250m d'une station.

In [ ]:
# Conversion explicite en format date
df_ready = gdf_final.copy()
df_ready['date_mutation'] = pd.to_datetime(gdf_final['date_mutation'])

df_ready = sd.prepare_did_data(df_ready)
results = sd.run_did_regression(df_ready)
print(results.summary())
sd.plot_did_trends(df_ready)

Le modèle de différence-en-différences révèle que l'ouverture de la ligne B du métro n'a pas eu d'impact statistiquement significatif sur les prix immobiliers à proximité des stations ($p\text{-value} = 0,86$). En effet, comme la ligne a été mise en service en 2022, cette date a été utilisée comme référence pour modéliser l'avant et l'après métro B. Cependant, la ligne a été annoncée au début des années 2000 pour des travaux commencés en 2014. Ainsi, le prix de l'immobilier dans les quartiers desservis par cette ligne a pu augmenter par anticipation bien avant les années 2020, année de début de notre base de données des valeurs foncières.

# Modèle de prédiction du prix 

### IMPORTANCE DES VARIABLES DANS LE MODÈLE :

In [ ]:

model_final, features_utilisees, le = preparer_et_entrainer(gdf_final)
importances = model_final.feature_importances_
fi = pd.Series(importances, index=features_utilisees).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#1D9E75' if 'metro' in f else '#888780' for f in fi.index]
fi.plot(kind='barh', ax=ax, color=colors)

ax.set_title("Importance des variables — Random Forest", fontsize=13)
ax.set_xlabel("Importance (réduction moyenne d'impureté)")
ax.axvline(x=0, color='black', linewidth=0.5)

for i, v in enumerate(fi):
    ax.text(v + 0.002, i, f"{v:.3f}", va='center', fontsize=9)

plt.tight_layout()
plt.show()

Les variables qui pèse le plus dans le modèle sont les distance à une station de métro ( A ou B ) et la surface du batiment. Puis vient la surface terrain. Ça parait plausible. 

In [ ]:

model_final, features_utilisees, le = preparer_et_entrainer(gdf_final)

# Bien réel observé (ses vraies distances actuelles aux lignes A et B)
bien_actuel = {
    'nombre_pieces_principales': 3,
    'surface_reelle_bati': 65,
    'surface_terrain': 0,
    'dist_metro_A': 1400,  # loin des deux lignes
    'dist_metro_B': 1400,
    'annee': 2026,
    'type_local': 'Appartement',
}

# Scénario : une nouvelle station A ouvre à 80m de ce bien

bien_nouvelle_station = bien_actuel.copy()
bien_nouvelle_station['dist_metro_A'] = 80  

prix_sans = predire_impact_nouvelle_station(model_final, bien_actuel, le)
prix_avec = predire_impact_nouvelle_station(model_final, bien_nouvelle_station, le)

plus_value = prix_avec - prix_sans
plus_value_pct = (plus_value / prix_sans) * 100

print(f"Prix estimé actuel               : {prix_sans:.0f} €/m²")
print(f"Prix estimé avec nouvelle station: {prix_avec:.0f} €/m²")
print(f"Plus-value estimée               : +{plus_value:.0f} €/m² ({plus_value_pct:.1f}%)")

Si une nouvelle station A est crée du jour au lendemain à 80m du bien immobilier témoin, il en résulterai une augmentation de +395€ du mètre carré.
On regarde la plus-value avec une nouvelle station B : 

In [ ]:

model_final, features_utilisees, le = preparer_et_entrainer(gdf_final)

# Bien réel observé (ses vraies distances actuelles aux lignes A et B)
bien_actuel = {
    'nombre_pieces_principales': 3,
    'surface_reelle_bati': 65,
    'surface_terrain': 0,
    'dist_metro_A': 1400,  # loin des deux lignes
    'dist_metro_B': 1400,
    'annee': 2026,
    'type_local': 'Appartement',
}

# Scénario : une nouvelle station B ouvre à 80m de ce bien
bien_nouvelle_station = bien_actuel.copy()
bien_nouvelle_station['dist_metro_B'] = 80 

prix_sans = predire_impact_nouvelle_station(model_final, bien_actuel, le)
prix_avec = predire_impact_nouvelle_station(model_final, bien_nouvelle_station, le)

plus_value = prix_avec - prix_sans
plus_value_pct = (plus_value / prix_sans) * 100

print(f"Prix estimé actuel               : {prix_sans:.0f} €/m²")
print(f"Prix estimé avec nouvelle station: {prix_avec:.0f} €/m²")
print(f"Plus-value estimée               : +{plus_value:.0f} €/m² ({plus_value_pct:.1f}%)")

Pour une nouvelle station B proche du bien témoin, il en résulterai une augmentation de 1360€ du mètre carré pour ce bien. 

## Heatmap Nouvelle station A proche de tous les bien à Rennes

In [ ]:
carte_plus_value(
    gdf_final=gdf_final,
    gdf_metro=gdf_metro,
    model=model_final,
    le=le,
    ligne='A',                       # 'A' ou 'B'
    distance_nouvelle_station=100,   # en mètres
)

On voit que certaine zone sont en rouge, c'est-à-dire que une nouvelle station à cet endroit aurait un effet negatif sur les prix de l'immobilier proche de la zone. Ça parait peu réaliste mais c'est probablement dû à comment le modèle a été entrainé donc le résultat n'est pas à prendre à la lette. Par ailleurs beaucoup d'autre zone sont en verte et donc l'effet d'une nouvelle station entrainerait une augmentation probable très prix de l'immobilier. 